In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys
sys.dont_write_bytecode = True
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(variable, '1')
NOTEBOOK_NAME = '05_01_task_fmri_hddm_associations.ipynb'
NOTEBOOK_RELATIVE_PATH = '05_01_task_fmri_hddm_associations.ipynb'
WORKING_DIRECTORY = Path.cwd().resolve()
NOTEBOOK_CANDIDATES = [Path(os.environ['A8_NOTEBOOK_PATH']).expanduser()] if 'A8_NOTEBOOK_PATH' in os.environ else [WORKING_DIRECTORY / NOTEBOOK_NAME, WORKING_DIRECTORY / NOTEBOOK_RELATIVE_PATH]
NOTEBOOK_PATH = next((p.resolve() for p in NOTEBOOK_CANDIDATES if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError('Set A8_NOTEBOOK_PATH to this notebook, or run with its containing directory as the working directory')
ANALYSIS_ROOT = NOTEBOOK_PATH.parent


In [ ]:
import numpy as u00_np
import pandas as u00_pd
from scipy import stats as u00_stats


In [ ]:
def u00_behavior_scored_trials():
    d = u00_behavior().copy()
    d['rt_valid'] = d.correct.eq(1) & d.rt_s.notna()
    for ((sid, cond), g) in d[d.task.eq('stroop')].groupby(['subject_id', 'condition']):
        r = g.loc[g.rt_valid, 'rt_s']
        sd = r.std(ddof=1)
        if len(r) > 1:
            d.loc[g.index, 'rt_valid'] &= d.loc[g.index, 'rt_s'].between(r.mean() - 3 * sd, r.mean() + 3 * sd)
    gng = d.task.eq('gonogo')
    d.loc[gng, 'rt_valid'] &= d.loc[gng, 'rt_s'].between(0.15, 0.5)
    return d


In [ ]:
def u00_behavior_subject_scores(d=None):
    d = u00_behavior_scored_trials() if d is None else d
    rows = []
    for (sid, g) in d.groupby('subject_id'):
        st = g[g.task.eq('stroop')]
        ng = g[g.task.eq('gonogo')]
        go = ng[ng.condition.eq('go')]
        no = ng[ng.condition.eq('nogo')]
        row = {'subject_id': sid}
        for c in ['congruent', 'neutral', 'incongruent']:
            b = st[st.condition.eq(c)]
            row['stroop_' + c + '_rt_ms'] = b.loc[b.rt_valid, 'rt_s'].mean() * 1000
            row['stroop_' + c + '_accuracy'] = b.correct.mean()
        row['stroop_rt_cost_ms'] = row['stroop_incongruent_rt_ms'] - row['stroop_congruent_rt_ms']
        row['stroop_accuracy_cost_pct'] = (row['stroop_congruent_accuracy'] - row['stroop_incongruent_accuracy']) * 100
        row['stroop_mean_rt_ms'] = st.loc[st.rt_valid, 'rt_s'].mean() * 1000
        row['gonogo_go_rt_ms'] = go.loc[go.rt_valid, 'rt_s'].mean() * 1000
        row['go_accuracy'] = go.correct.mean()
        row['nogo_accuracy'] = no.correct.mean()
        row['gonogo_commission_pct'] = (1 - row['nogo_accuracy']) * 100
        row['gonogo_omission_pct'] = (1 - row['go_accuracy']) * 100
        row['gonogo_dprime'] = u00_stats.norm.ppf((go.correct.sum() + 0.5) / (len(go) + 1)) - u00_stats.norm.ppf(((1 - no.correct).sum() + 0.5) / (len(no) + 1))
        row['accuracy_difference'] = row['nogo_accuracy'] - row['go_accuracy']
        rows.append(row)
    result = u00_pd.DataFrame(rows)
    if not u00_np.isfinite(result.drop(columns='subject_id').to_numpy(float)).all():
        raise ValueError('Insufficient valid trials for a behavioral measure')
    return result


In [ ]:
from pathlib import Path as u01_Path
from pathlib import Path as u01_Path
import json as u01_json, re as u01_re
import numpy as u01_np
import pandas as u01_pd


In [ ]:
def u01_ddm_fit_hddm(samples=15000, burn=5000, chains=4, seed=20261021, data=None, output_dir=None, model_check=False, ppc_samples=100, export_long=False):
    import hddm as hddm
    u01_ddm_validate_hddm_backend(hddm)
    if data is not None and (not model_check):
        raise ValueError('Explicit in-memory data are reserved for labelled model checks')
    d = u01_observed_hddm_data() if data is None else data.copy()
    d['subj_idx'] = d.subj_idx.astype(str)
    ids = set(u01_participants().subject_id)
    if set(d.subj_idx) != ids or set(d.task) != {'stroop', 'gonogo'}:
        raise ValueError('HDDM input cohort or task mismatch')
    if not d.response.isin([0, 1]).all() or not u01_np.isfinite(d.rt).all() or (d.rt <= 0).any():
        raise ValueError('Positive observed RT in seconds; response codes 0/1; missing RT encoded 999')
    if burn >= samples or chains < 2:
        raise ValueError('Multiple chains and samples > burn are required')
    out = u01_Path(output_dir) if output_dir is not None else u01_ddm_OUT
    (out / 'tables').mkdir(parents=True, exist_ok=True)
    fit_state = None
    if not model_check:
        import hashlib as hashlib
        fit_state = dict(status='running', samples=samples, burn=burn, chains=chains, ppc_samples=ppc_samples, export_long=export_long, model_code_sha256=u01_hddm_source_signature(), input_sha256={name: hashlib.sha256(u01_input_path(name).read_bytes()).hexdigest() for name in ['Participants.csv', 'Behavior_data.csv', 'HDDM_data.csv']})
        u01_atomic_record(out / 'fit_state.json', fit_state)
    long_path = out / 'tables/HDDM_posterior_draws.csv'
    wrote = False
    means = []
    diagrows = []
    for task in ['stroop', 'gonogo']:
        g = d[d.task.eq(task)].copy().reset_index(drop=True)
        expected = {'congruent', 'neutral', 'incongruent'} if task == 'stroop' else {'go', 'nogo'}
        if set(g.condition) != expected or set(g.subj_idx) != ids:
            raise ValueError('Missing task subjects or conditions')
        if not g.groupby('subj_idx').condition.apply(lambda x: set(x) == expected).all():
            raise ValueError('A subject lacks a model condition')
        if task == 'stroop':
            if not g.stimulus.isin([0, 1]).all() or g.rt.ge(998).any():
                raise ValueError('Stroop needs observed RT and actual color-boundary coding')
        elif (g.response.eq(0) != g.rt.eq(999)).any() or g.loc[g.response.eq(1), 'rt'].ge(998).any():
            raise ValueError('Go/No-Go lower boundary = nonresponse and RT 999; upper boundary = observed Go response')
        task_traces = []
        for chain in range(chains):
            u01_np.random.seed(seed + chain + (10000 if task == 'gonogo' else 0))
            kwargs = dict(depends_on={'v': 'condition'}, include=['a', 'v', 't', 'st'] + (['z'] if task == 'gonogo' else []), group_only_nodes=['st'], p_outlier=0.05)
            model = hddm.HDDMStimCoding(g, stim_col='stimulus', split_param='v', **kwargs) if task == 'stroop' else hddm.HDDM(g, **kwargs)
            model.find_starting_values()
            model.sample(samples, burn=burn, dbname=str(out / f'{task}_chain{chain}.db'), db='pickle')
            model.save(str(out / f'{task}_chain{chain}.model'))
            traces = model.get_traces()
            if traces.empty or not u01_np.isfinite(traces.to_numpy(float)).all():
                raise RuntimeError('Missing/nonfinite MCMC draws')
            traces.to_csv(out / f'{task}_chain{chain}_draws.csv', index=False)
            task_traces.append(traces)
            for node in traces:
                match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(node))
                if match:
                    (par, condition, sid) = match.groups()
                    parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
                    if sid not in ids or parameter not in u01_PARAMS:
                        raise ValueError('Unmapped subject parameter: ' + str(node))
                else:
                    (sid, parameter) = ('__group__', task + '_group_' + str(node))
                if export_long:
                    block = u01_pd.DataFrame(dict(task=task, chain=chain, draw=u01_np.arange(len(traces)), subject_id=sid, parameter=parameter, value=traces[node].to_numpy()))
                    block.to_csv(long_path, index=False, mode='a' if wrote else 'w', header=not wrote)
                    wrote = True
            if ppc_samples:
                ppc = u01_ddm_posterior_predictive_samples(model, task, ppc_samples, seed + chain)
                ppc.to_csv(out / f'{task}_chain{chain}_PPC.csv')
            del model
        if any((list(tr.columns) != list(task_traces[0].columns) or len(tr) != len(task_traces[0]) for tr in task_traces)):
            raise ValueError('MCMC chain node or draw mismatch')
        for node in task_traces[0]:
            match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(node))
            values = u01_np.stack([tr[node].to_numpy() for tr in task_traces])
            if match:
                (par, condition, sid) = match.groups()
                parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
                means.append(dict(subject_id=sid, parameter=parameter, value=values.mean()))
            if model_check:
                import arviz as az
                diagrows.append(dict(task=task, node=node, r_hat=float(az.rhat(values, method='rank')), ess_bulk=float(az.ess(values, method='bulk')), ess_tail=float(az.ess(values, method='tail'))))
    mean_table = u01_pd.DataFrame(means).pivot(index='subject_id', columns='parameter', values='value').reset_index()
    if set(u01_PARAMS) - set(mean_table) or len(mean_table) != len(ids):
        raise ValueError('Incomplete subject posterior means')
    mean_table['data_status'] = 'MODEL_CHECK_SIMULATION_REFITTED_NOT_EMPIRICAL' if model_check else 'FITTED_FROM_SUPPLIED_OBSERVATIONS'
    mean_table.to_csv(out / 'tables/HDDM_parameters.csv', index=False)
    if model_check:
        diagnostics = u01_pd.DataFrame(diagrows)
        diagnostics['diagnostics_pass'] = diagnostics.r_hat.lt(1.01) & diagnostics.ess_bulk.gt(400) & diagnostics.ess_tail.gt(400)
        diagnostics.to_csv(out / 'tables/Recovery_fit_diagnostics.csv', index=False)
        (out / 'MODEL_CHECK_ONLY.json').write_text(u01_json.dumps(dict(status='PARAMETER_RECOVERY_SIMULATION_REFIT_NOT_EMPIRICAL', seed=seed, samples=samples, burn=burn, chains=chains)))
    else:
        u01_provenance('hddm_fit', ['Participants.csv', 'Behavior_data.csv', 'HDDM_data.csv'], settings=dict(samples=samples, burn=burn, chains=chains, ppc_samples=ppc_samples, seed=seed, p_outlier=0.05))
        fit_state['status'] = 'complete'
        u01_atomic_record(out / 'fit_state.json', fit_state)
    return out


In [ ]:
def u01_ddm_validate_hddm_backend(hddm):
    import inspect as inspect
    from scipy import stats as stats
    if str(hddm.__version__) != '1.0.1':
        raise RuntimeError('Manuscript requires HDDM 1.0.1; install and validate the specified environment')
    source = inspect.getsource(hddm.likelihoods.generate_wfpt_stochastic_class)
    tree = __import__('ast').parse(source)
    fn = next((n for n in tree.body[0].body if isinstance(n, __import__('ast').FunctionDef) and n.name == 'wfpt_like'))
    namespace = dict(hddm=hddm, np=u01_np, stats=stats, wp=dict(err=0.0001, n_st=2, n_sz=2, use_adaptive=1, simps_err=0.001, w_outlier=0.1))
    module = __import__('ast').Module(body=[fn], type_ignores=[])
    exec(compile(module, 'installed_hddm_missing_rt_check', 'exec'), namespace)
    for v in [-1.0, 0.0, 1.0]:
        expected = 0.4 if v == 0 else u01_np.expm1(-2 * 1.2 * 0.4 * v) / u01_np.expm1(-2 * 1.2 * v)
        for boundary in [-1, 1]:
            val = namespace['wfpt_like'](u01_pd.DataFrame({'rt': [boundary * 999.0]}), v, 0, 1.2, 0.4, 0, 0.2, 0.05, p_outlier=0.05)
            prob = expected if boundary == 1 else 1 - expected
            if not u01_np.isclose(val, u01_np.log(prob), rtol=1e-06, atol=1e-08):
                raise RuntimeError('Installed HDDM does not implement the documented implicit-boundary missing-RT likelihood')
    return True


In [ ]:
def u01_ddm_recovery_sample(model, task, position):
    from kabuki.analyze import _parents_to_random_posterior_sample as _parents_to_random_posterior_sample
    traces = model.get_traces()
    if not 0 <= position < len(traces):
        raise ValueError('Posterior draw index outside fitted chain')
    truth = []
    for (name, value) in traces.iloc[position].items():
        match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(name))
        if match:
            (par, condition, sid) = match.groups()
            parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
            truth.append(dict(subject_id=sid, parameter=parameter, generating_value=float(value)))
    generated = []
    for (name, record) in model.iter_observeds():
        node = record['node']
        _parents_to_random_posterior_sample(node, pos=position)
        sample = node.random(add_outliers=True)
        observed = model.data.loc[node.value.index].copy()
        if len(sample) != len(observed):
            raise ValueError('Recovery generator changed trial counts')
        signed_rt = sample['rt'].to_numpy(float)
        observed['response'] = (signed_rt > 0).astype(int)
        observed['rt'] = u01_np.abs(signed_rt)
        if task == 'gonogo':
            observed.loc[observed.response.eq(0), 'rt'] = 999.0
        observed['task'] = task
        generated.append(observed)
    data = u01_pd.concat(generated).sort_index().reset_index(drop=True)
    if len(data) != len(model.data):
        raise ValueError('Recovery generation did not cover all observed trials')
    if len(truth) != len(u01_participants()) * (5 if task == 'stroop' else 5):
        raise ValueError('Incomplete generating subject parameters')
    return (data, u01_pd.DataFrame(truth))


In [ ]:
def u01_ddm_posterior_predictive_samples(model, task, samples=100, seed=20261021):
    rng = u01_np.random.default_rng(seed)
    rows = []
    original = model.data.sort_index().reset_index(drop=True)
    for sample in range(samples):
        position = int(rng.integers(len(model.get_traces())))
        (generated, _) = u01_ddm_recovery_sample(model, task, position)
        if not generated[['subj_idx', 'condition', 'stimulus']].equals(original[['subj_idx', 'condition', 'stimulus']]):
            raise ValueError('Posterior prediction lost original trial alignment')
        block = original.copy()
        block['rt_sampled'] = u01_np.where(generated.response.eq(1), generated.rt, -generated.rt)
        block['response_sampled'] = generated.response.to_numpy()
        block['data_status'] = 'POSTERIOR_PREDICTIVE_SIMULATION_WITH_APPENDED_OBSERVATIONS'
        block['sample'] = sample
        block['posterior_draw'] = position
        rows.append(block)
    return u01_pd.concat(rows, ignore_index=True)


In [ ]:
from pathlib import Path as u07_Path
import sys as u07_sys
from IPython.display import display as u07_display, Image as u07_Image
from pathlib import Path as u07_Path
from dataclasses import dataclass as u07_dataclass
import os as u07_os, sys as u07_sys, json as u07_json, hashlib as u07_hashlib, warnings as u07_warnings
for u07_key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u07_os.environ.setdefault(u07_key, '1')
import numpy as u07_np
import pandas as u07_pd
from scipy import stats as u07_stats, integrate as u07_integrate, special as u07_special
from sklearn.model_selection import KFold as u07_KFold
from statsmodels.stats.multitest import multipletests as u07_multipletests
import sys as u07_sys
from pathlib import Path as u07_Path
u07_hddm_ROIS = ['shared_SMA', 'shared_rAI_FO', 'shared_lAI', 'stroop_lIPS', 'gonogo_rIFG_PMC']
u07_hddm_GENERAL = ['shared_v', 'shared_a', 'shared_t']
u07_hddm_SPECIFIC = ['stroop_specific', 'gonogo_specific']
u07_hddm_TERMS = u07_hddm_GENERAL + u07_hddm_SPECIFIC
u07_hddm_FC = ['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge']
u07_hddm_COV = ['age', 'sex', 'education', 'mean_fd_mm']
u07_hddm_REQUIRED = ['stroop_a', 'stroop_v_congruent', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u07_hddm_STATUS = 'OBSERVED_INPUTS: fitted parameters plus measured behavior/MRI'


In [ ]:
@u07_dataclass(frozen=True)
class u07_hddm_Settings:
    bootstrap: int = 5000
    difference_bootstrap: int = 3000
    cv_repeats: int = 100
    permutations: int = 1000
    outer_folds: int = 5
    inner_folds: int = 5
    alpha_grid: tuple = (0.01, 0.1, 1.0, 10.0, 100.0)
    seed: int = 20261018
    equivalence_r: float = 0.3


In [ ]:
u07_hddm_CFG = u07_hddm_Settings()


In [ ]:
def u07_hddm_z(x):
    x = u07_np.asarray(x, float)
    sd = x.std(axis=0, ddof=1)
    if u07_np.any(sd < 1e-10):
        raise ValueError('Constant variable; inspect data')
    return (x - x.mean(axis=0)) / sd


In [ ]:
def u07_hddm_boolean(x):
    if x.dtype == bool:
        return x
    return x.astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False}).astype('boolean')


In [ ]:
def u07_hddm_bh(p):
    p = u07_np.asarray(p, float)
    o = u07_np.full(len(p), u07_np.nan)
    ok = u07_np.isfinite(p)
    if ok.any():
        o[ok] = u07_multipletests(p[ok], method='fdr_bh')[1]
    return o


In [ ]:
def u07_hddm_features(d):
    d = d.copy()
    for (p, a, b) in [('v', 'stroop_v_congruent', 'gonogo_v_go'), ('a', 'stroop_a', 'gonogo_a'), ('t', 'stroop_t', 'gonogo_t')]:
        d['shared_' + p] = u07_hddm_z((u07_hddm_z(d[a]) + u07_hddm_z(d[b])) / 2)
    for (task, target, base) in [('stroop', 'stroop_v_incongruent', 'stroop_v_congruent'), ('gonogo', 'gonogo_v_nogo', 'gonogo_v_go')]:
        X = u07_np.c_[u07_np.ones(len(d)), d[base]]
        d[task + '_specific'] = u07_hddm_z(d[target].to_numpy() - X @ u07_np.linalg.lstsq(X, d[target], rcond=None)[0])
    return d


In [ ]:
def u07_hddm_read_real():
    d = u07_posterior()
    return u07_hddm_features(d.sort_values('subject_id').reset_index(drop=True))


In [ ]:
def u07_hddm_load_real_bundle():
    verify_imaging = u19_main
    verify_imaging()
    d = u07_hddm_read_real()
    if d is None:
        return None
    behavior = u10_brain_behavior_behavior()
    d = d.merge(behavior, on='subject_id', validate='one_to_one')
    if len(d) != u07_EXPECTED_N:
        raise ValueError('Demographics/behavior merge lost subjects')
    base = u07_hddm_MRI / '2.task_fmri/other_outputs'
    roi = u07_pd.read_csv(base / 'independent_ALE_ROI_ALE_3D_mean_t.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    pattern = u07_pd.read_csv(base / 'independent_ALE_ROI_unsmoothed_patterns.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    motion = u07_pd.read_csv(base / 'observed_inputs_and_model_validation.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    aggregate_task_motion = u15_aggregate_task_motion
    motion = aggregate_task_motion(motion)
    task = roi.merge(motion, on=['subject_id', 'task'], validate='many_to_one').merge(d, on='subject_id', validate='many_to_one')
    task = task[task.coverage >= 0.8].copy()
    rest = u07_pd.read_csv(u07_hddm_MRI / '1.resting_state/other_outputs/independent_ALE_ROI_connectivity_subject_level.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    rest['GSR'] = u07_hddm_boolean(rest.GSR)
    rest['strict_motion'] = u07_hddm_boolean(rest.strict_motion)
    rest = rest[rest.roi_definition.eq('ALE_3D')].merge(d, on='subject_id', validate='many_to_one')
    pattern = pattern[pattern.coverage >= 0.8].copy()
    pattern = pattern.merge(d, on='subject_id', validate='many_to_one')
    both = motion.groupby('subject_id').agg(mean_fd_mm=('mean_fd_mm', 'mean'), strict_motion=('strict_motion', 'all')).reset_index()
    pattern = pattern.merge(both, on='subject_id', validate='many_to_one')
    return dict(task=task, rest=rest, pattern=pattern, status=u07_hddm_STATUS)


In [ ]:
def u07_hddm_save(df, folder, name, status):
    folder = u07_Path(folder)
    (folder / 'tables').mkdir(exist_ok=True, parents=True)
    df = df.copy()
    df.insert(0, 'data_status', status)
    if 'N' in df:
        df.insert(1, 'N_planned', u07_EXPECTED_N)
        df.insert(2, 'N_not_in_this_analysis', u07_EXPECTED_N - df.N)
    prefix = 'REAL-'
    path = folder / 'tables' / f'{prefix}{name}.csv'
    df.to_csv(path, index=False, encoding='utf-8-sig', float_format='%.10g')
    return path


In [ ]:
def u07_hddm_model(data, outcome, cols):
    g = data[['subject_id', outcome] + cols].replace([u07_np.inf, -u07_np.inf], u07_np.nan).dropna().copy()
    n = len(g)
    X = g[cols].to_numpy(float)
    y = u07_hddm_z(g[outcome].to_numpy(float))
    for (i, c) in enumerate(cols):
        if c != 'sex':
            X[:, i] = u07_hddm_z(X[:, i])
    X = u07_np.c_[u07_np.ones(n), X]
    rank = u07_np.linalg.matrix_rank(X)
    if rank != X.shape[1] or n <= rank + 3:
        raise ValueError('Insufficient observations/full rank for ' + outcome)
    beta = u07_np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    df = n - rank
    sse = e @ e
    sst = u07_np.sum((y - y.mean()) ** 2)
    se = u07_np.sqrt(u07_np.diag(u07_np.linalg.inv(X.T @ X)) * sse / df)
    return dict(g=g, X=X, y=y, beta=beta, se=se, df=df, r2=1 - sse / sst, sse=sse, cols=cols)


In [ ]:
def u07_hddm_zero_evidence(r, n, k, y, X_full, X_null, cfg=u07_hddm_CFG):
    eq = u07_equivalence(r, n, k, cfg.equivalence_r)
    bf = u07_jzs_nested(y, X_full, X_null)
    return dict(partial_r=r, r90_low=eq['r90_low'], r90_high=eq['r90_high'], p_TOST_r30=eq['p_TOST'], BF10_conditional_JZS=bf['BF10'], BF01_conditional_JZS=bf['BF01'], JZS_prior=bf['JZS_prior'])


In [ ]:
def u07_hddm_coefficients(fit, terms, cfg=u07_hddm_CFG, assess_zero=False):
    rows = []
    n = len(fit['y'])
    for term in terms:
        i = fit['cols'].index(term) + 1
        b = fit['beta'][i]
        se = fit['se'][i]
        tt = b / se
        ci = b + u07_np.array([-1, 1]) * u07_stats.t.ppf(0.975, fit['df']) * se
        row = dict(parameter=term, N=n, df=fit['df'], beta=b, CI_low=ci[0], CI_high=ci[1], p=2 * u07_stats.t.sf(abs(tt), fit['df']))
        if assess_zero:
            row.update(u07_hddm_zero_evidence(tt / u07_np.sqrt(tt * tt + fit['df']), n, len(fit['cols']) - 1, fit['y'], fit['X'][:, 1:], u07_np.delete(fit['X'], i, axis=1)[:, 1:], cfg))
        rows.append(row)
    return rows


In [ ]:
def u07_hddm_boot_models(data, outcome, models, B, seed):
    cols = list(dict.fromkeys([v for mm in models for v in mm]))
    g = data[['subject_id', outcome] + cols].replace([u07_np.inf, -u07_np.inf], u07_np.nan).dropna().reset_index(drop=True)
    n = len(g)
    rng = u07_np.random.default_rng(seed)
    r2 = []
    betas = []
    for start in range(0, B, 200):
        take = rng.integers(0, n, size=(min(200, B - start), n))
        Y = g[outcome].to_numpy(float)[take]
        my = Y.mean(axis=1, keepdims=True)
        sy = Y.std(axis=1, ddof=1, keepdims=True)
        if not u07_np.isfinite(sy).all() or u07_np.any(sy <= 1e-12):
            raise ValueError('Degenerate bootstrap outcome; inspect observed variation')
        Y = (Y - my) / sy
        rbatch = []
        bbatch = []
        for mm in models:
            V = g[mm].to_numpy(float)[take]
            mu = V.mean(axis=1, keepdims=True)
            sd = V.std(axis=1, ddof=1, keepdims=True)
            V = (V - mu) / u07_np.where(sd > 1e-10, sd, 1)
            if 'sex' in mm:
                V[:, :, mm.index('sex')] = g.sex.to_numpy(float)[take]
            X = u07_np.concatenate([u07_np.ones((*take.shape, 1)), V], axis=2)
            beta = (u07_np.linalg.pinv(X) @ Y[:, :, None])[:, :, 0]
            res = Y - u07_np.einsum('bnp,bp->bn', X, beta)
            rbatch.append(1 - u07_np.sum(res ** 2, axis=1) / u07_np.sum(Y ** 2, axis=1))
            bbatch.append(beta)
        r2.append(u07_np.stack(rbatch, axis=1))
        betas.append(bbatch)
    return (u07_np.concatenate(r2), [u07_np.concatenate([b[j] for b in betas]) for j in range(len(models))])


In [ ]:
def u07_hddm_variants(bundle, modality):
    d = bundle[modality]
    if modality == 'rest':
        for gsr in [False, True]:
            for strict in [False, True]:
                g = d[d.GSR.eq(gsr)]
                g = g[g.strict_motion] if strict else g
                yield (f"{('GSR' if gsr else 'noGSR')}_{('strict' if strict else 'all')}", g)
    else:
        yield ('all', d)
        yield ('strict', d[d.strict_motion])


In [ ]:
def u07_hddm_partial(g, x, y, covs, cfg=u07_hddm_CFG, assess_zero=False):
    a = g[[x, y] + covs].replace([u07_np.inf, -u07_np.inf], u07_np.nan).dropna().to_numpy(float)
    n = len(a)
    C = u07_np.c_[u07_np.ones(n), a[:, 2:]]
    k = u07_np.linalg.matrix_rank(C) - 1
    if n <= k + 3:
        raise ValueError('Insufficient N for partial correlation')
    e = a[:, :2] - C @ u07_np.linalg.lstsq(C, a[:, :2], rcond=None)[0]
    r = u07_np.corrcoef(e.T)[0, 1]
    df = n - k - 2
    tt = r * u07_np.sqrt(df / max(1 - r * r, 1e-12))
    ci = u07_np.tanh(u07_np.arctanh(u07_np.clip(r, -0.999999, 0.999999)) + u07_np.array([-1, 1]) * u07_stats.norm.ppf(0.975) / u07_np.sqrt(n - k - 3))
    row = dict(N=n, df=df, r=r, p=2 * u07_stats.t.sf(abs(tt), df), CI95_low=ci[0], CI95_high=ci[1], CI_method='Fisher-z with nuisance-rank adjustment')
    if assess_zero:
        row.update(u07_hddm_zero_evidence(r, n, k, a[:, 1], u07_np.c_[a[:, 2:], a[:, 0]], a[:, 2:], cfg))
    return row


In [ ]:
def u07_hddm_task_primary(bundle, cfg=u07_hddm_CFG):
    all_results = []
    differences = []
    for (variant, d) in u07_hddm_variants(bundle, 'task'):
        for own in [False, True]:
            general = u07_hddm_GENERAL
            rows = []
            increments = []
            variance = []
            for ((task, roi), raw) in d.groupby(['task', 'roi_id']):
                g = raw.copy()
                if own:
                    for (p, col) in [('v', 'stroop_v_congruent' if task == 'stroop' else 'gonogo_v_go'), ('a', task + '_a'), ('t', task + '_t')]:
                        g['shared_' + p] = u07_hddm_z(g[col])
                cols = u07_hddm_COV + general + u07_hddm_SPECIFIC
                f = u07_hddm_model(g, 'mean_voxel_t', cols)
                reduced = u07_hddm_model(f['g'], 'mean_voxel_t', u07_hddm_COV + general)
                for row in u07_hddm_coefficients(f, ['shared_v'] + u07_hddm_SPECIFIC, cfg):
                    rows.append(dict(task=task, roi_id=roi, **row))
                models = [u07_hddm_COV, u07_hddm_COV + general, u07_hddm_COV + u07_hddm_SPECIFIC, cols]
                (br, _) = u07_hddm_boot_models(g, 'mean_voxel_t', models, cfg.bootstrap, cfg.seed)
                delta = f['r2'] - reduced['r2']
                bdelta = br[:, 3] - br[:, 1]
                F = (reduced['sse'] - f['sse']) / 2 / (f['sse'] / f['df'])
                ci = u07_np.quantile(bdelta, [0.025, 0.975])
                increments.append(dict(task=task, roi_id=roi, N=len(f['y']), R2_step1=reduced['r2'], R2_step2=f['r2'], delta_R2=delta, CI_low=ci[0], CI_high=ci[1], F_change=F, df1=2, df2=f['df'], p_F_change=u07_stats.f.sf(F, 2, f['df']), bootstrap_draws=cfg.bootstrap))
                if roi in u07_hddm_ROIS[:3]:
                    r0 = u07_hddm_model(g, 'mean_voxel_t', u07_hddm_COV)['r2']
                    rs = u07_hddm_model(g, 'mean_voxel_t', u07_hddm_COV + u07_hddm_SPECIFIC)['r2']
                    ig = reduced['r2'] - r0
                    isp = rs - r0
                    it = f['r2'] - r0
                    vals = {'unique_general': (it - isp, br[:, 3] - br[:, 2]), 'unique_specific': (it - ig, br[:, 3] - br[:, 1]), 'common': (ig + isp - it, br[:, 1] + br[:, 2] - br[:, 3] - br[:, 0])}
                    for (component, (estimate, draw)) in vals.items():
                        ci = u07_np.quantile(draw, [0.025, 0.975])
                        variance.append(dict(task=task, roi_id=roi, N=len(f['y']), component=component, variance_fraction=estimate, CI_low=ci[0], CI_high=ci[1], total_increment=it, bootstrap_draws=cfg.bootstrap))
            tab = u07_pd.DataFrame(rows)
            tab['q_FDR_30'] = u07_hddm_bh(tab.p)
            inc = u07_pd.DataFrame(increments)
            inc['q_FDR_10_secondary'] = u07_hddm_bh(inc.p_F_change)
            label = variant + ('_own_general' if own else '_shared_general')
            tab['variant'] = label
            all_results.append(tab)
            u07_hddm_save(tab, u07_hddm_TASK, 'task_five_ROI_hierarchical_regression_coefficients_' + label, bundle['status'])
            u07_hddm_save(inc, u07_hddm_TASK, 'task_five_ROI_specific_incremental_R2_' + label, bundle['status'])
            u07_hddm_save(u07_pd.DataFrame(variance), u07_hddm_TASK, 'shared_ROI_variance_decomposition_' + label, bundle['status'])
        for own in [False, True]:
            diff = []
            for roi in u07_hddm_ROIS:
                if roi in u07_hddm_ROIS[:3]:
                    spec = [('stroop', 'shared_v', 'gonogo', 'shared_v')]
                elif roi == 'stroop_lIPS':
                    spec = [('stroop', 'stroop_specific', 'stroop', 'gonogo_specific')]
                else:
                    spec = [('gonogo', 'gonogo_specific', 'gonogo', 'stroop_specific')]
                for (ta, pa, tb, pb) in spec:
                    a = d[d.roi_id.eq(roi) & d.task.eq(ta)]
                    b = d[d.roi_id.eq(roi) & d.task.eq(tb)]
                    ids = sorted(set(a.subject_id) & set(b.subject_id))
                    a = a.set_index('subject_id').loc[ids].reset_index()
                    b = b.set_index('subject_id').loc[ids].reset_index()
                    if own:
                        for (dd, task_) in [(a, ta), (b, tb)]:
                            for (pp, cc) in [('v', 'stroop_v_congruent' if task_ == 'stroop' else 'gonogo_v_go'), ('a', task_ + '_a'), ('t', task_ + '_t')]:
                                dd['shared_' + pp] = u07_hddm_z(dd[cc])
                    cols = u07_hddm_COV + u07_hddm_GENERAL + u07_hddm_SPECIFIC
                    i = cols.index(pa) + 1
                    j = cols.index(pb) + 1
                    obs = u07_hddm_model(a, 'mean_voxel_t', cols)['beta'][i] - u07_hddm_model(b, 'mean_voxel_t', cols)['beta'][j]
                    (_, ba) = u07_hddm_boot_models(a, 'mean_voxel_t', [cols], cfg.difference_bootstrap, cfg.seed)
                    (_, bb) = u07_hddm_boot_models(b, 'mean_voxel_t', [cols], cfg.difference_bootstrap, cfg.seed)
                    draw = ba[0][:, i] - bb[0][:, j]
                    ci = u07_np.quantile(draw, [0.025, 0.975])
                    diff.append(dict(roi_id=roi, comparison=f'{ta}:{pa} minus {tb}:{pb}', N=len(ids), delta_beta=obs, CI_low=ci[0], CI_high=ci[1], bootstrap_draws=cfg.difference_bootstrap, inference='Descriptive paired bootstrap CI; no equivalence from nonsignificant difference'))
            dif = u07_pd.DataFrame(diff)
            u07_hddm_save(dif, u07_hddm_TASK, 'task_ROI_paired_coefficient_difference_' + variant + ('_own_general' if own else '_shared_general'), bundle['status'])
            differences.append(dif)
    return u07_pd.concat(all_results, ignore_index=True)


In [ ]:
def u07_hddm_task_secondary(bundle, cfg=u07_hddm_CFG):
    tables = []
    for (variant, d) in u07_hddm_variants(bundle, 'task'):
        rows = []
        for ((task, roi), g) in d[d.roi_id.isin(u07_hddm_ROIS[:3])].groupby(['task', 'roi_id']):
            other = 'gonogo' if task == 'stroop' else 'stroop'
            own = ['stroop_v_congruent', 'stroop_a', 'stroop_t'] if task == 'stroop' else ['gonogo_v_go', 'gonogo_a', 'gonogo_t']
            f = u07_hddm_model(g, 'mean_voxel_t', u07_hddm_COV + own + [other + '_specific'])
            row = u07_hddm_coefficients(f, [other + '_specific'], cfg, assess_zero=True)[0]
            rows.append(dict(target_task=task, roi_id=roi, predictor_task=other, **row))
        tab = u07_pd.DataFrame(rows)
        tab['q_FDR_6'] = u07_hddm_bh(tab.p)
        tab['variant'] = variant
        u07_hddm_save(tab, u07_hddm_TASK, 'cross_task_specific_component_predicts_other_task_activation_' + variant, bundle['status'])
        tables.append(tab)
    for (variant, d) in u07_hddm_variants(bundle, 'pattern'):
        rows = []
        for (roi, g) in d.groupby('roi_id'):
            for par in ['shared_v'] + u07_hddm_SPECIFIC:
                row = u07_hddm_partial(g, par, 'fisher_z', u07_hddm_COV, cfg, assess_zero=False)
                cols = [par, 'fisher_z'] + u07_hddm_COV
                a = g[cols].dropna().to_numpy(float)
                n = len(a)
                rng = u07_np.random.default_rng(cfg.seed)
                vals = []
                for start in range(0, cfg.bootstrap, 200):
                    ix = rng.integers(0, n, size=(min(200, cfg.bootstrap - start), n))
                    v = a[ix]
                    C = u07_np.concatenate([u07_np.ones((*ix.shape, 1)), v[:, :, 2:]], axis=2)
                    E = v[:, :, :2] - C @ (u07_np.linalg.pinv(C) @ v[:, :, :2])
                    E -= E.mean(axis=1, keepdims=True)
                    vals.extend(u07_np.sum(E[:, :, 0] * E[:, :, 1], axis=1) / u07_np.sqrt(u07_np.sum(E[:, :, 0] ** 2, axis=1) * u07_np.sum(E[:, :, 1] ** 2, axis=1)))
                ci = u07_np.quantile(vals, [0.025, 0.975])
                rows.append(dict(roi_id=roi, parameter=par, CI_low=ci[0], CI_high=ci[1], bootstrap_draws=cfg.bootstrap, **row))
        tab = u07_pd.DataFrame(rows)
        tab['q_FDR_9'] = u07_hddm_bh(tab.p)
        u07_hddm_save(tab, u07_hddm_TASK, 'shared_ROI_pattern_similarity_and_parameters_' + variant, bundle['status'])
    return u07_pd.concat(tables, ignore_index=True)


In [ ]:
def u07_hddm_figures(bundle, modality, cfg=u07_hddm_CFG):
    import matplotlib as matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import TwoSlopeNorm as TwoSlopeNorm
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 8, 'axes.spines.right': False, 'axes.spines.top': False, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'figure.facecolor': 'white'})
    compact_roi_names = {'shared_SMA': 'SMA', 'shared_rAI_FO': 'Right AI/FO', 'shared_lAI': 'Left AI', 'stroop_lIPS': 'Left IPS', 'gonogo_rIFG_PMC': 'Right IFG/PMC'}
    compact_task_names = {'stroop': 'Stroop', 'gonogo': 'Go/No-Go'}
    compact_term_names = {'shared_v': 'General v', 'stroop_specific': 'S residual', 'gonogo_specific': 'G residual'}
    folder = u07_hddm_REST if modality == 'rest' else u07_hddm_TASK
    prefix = 'REAL-'
    qa = u07_Path(u07_os.environ.get('A8_FIGURE_QA_DIR', u07_RESULTS / 'figure_qa'))
    qa.mkdir(parents=True, exist_ok=True)
    (folder / 'image').mkdir(exist_ok=True, parents=True)

    def add_estimate_labels(ax, values, upper_bounds, rows, lower_bounds=None):
        (low, high) = ax.get_xlim()
        span = high - low
        text_x = u07_np.asarray(upper_bounds, float) + 0.025 * span
        left_extent = low
        for (index, (value, xpos, ypos)) in enumerate(zip(values, text_x, rows)):
            align = 'left'
            if modality == 'task' and lower_bounds is not None and (float(value) < 0) and (float(u07_np.asarray(upper_bounds)[index]) < 0):
                xpos = float(u07_np.asarray(lower_bounds)[index]) - 0.025 * span
                left_extent = min(left_extent, xpos - 0.2 * span)
                align = 'right'
            ax.text(xpos, ypos, f'{float(value):.3f}', ha=align, va='center', fontsize=5.8 if modality == 'task' else 6.5, color='#222222')
        ax.set_xlim(left_extent, max(high, float(u07_np.max(text_x)) + 0.18 * span))

    def export(fig, stem):
        import textwrap
        from matplotlib.ticker import MaxNLocator
        compact_rest = modality == 'rest' and stem == 'rest_connectivity_and_computational_components'
        compact_task = modality == 'task'
        for ax in fig.axes:
            ax.tick_params(labelsize=5.8 if compact_task else 6 if compact_rest else 6.5)
            ax.xaxis.label.set_fontsize(6.3 if compact_task else 6.5 if compact_rest else 7)
            ax.yaxis.label.set_fontsize(6.3 if compact_task else 6.5 if compact_rest else 7)
            label = ax.get_xlabel()
            if compact_task:
                ax.set_xlabel(textwrap.fill(label, width=48), fontsize=6.3)
                ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
            elif len(label) > 40:
                ax.set_xlabel(textwrap.fill(label, width=40), fontsize=7)
            if ax.get_title():
                ax.set_title(textwrap.fill(ax.get_title(), width=46 if compact_task else 43 if compact_rest else 70), fontsize=7.2 if compact_task else 7.5 if compact_rest else 8, fontweight='bold' if compact_task or compact_rest else 'normal', pad=5)
        fig.tight_layout(pad=0.35 if compact_rest or compact_task else 0.6, rect=[0, 0.01, 1, 0.99])
        path = folder / 'image' / (prefix + stem + '.png')
        fig.savefig(path, dpi=600, bbox_inches='tight', pad_inches=0.03)
        if u07_os.environ.get('A8_FIGURE_QA_DIR'):
            fig.savefig(qa / (prefix + stem + '.pdf'), bbox_inches='tight', pad_inches=0.03)
        plt.close(fig)
    bias_path = folder / 'tables' / (prefix + 'starting_point_bias_sensitivity_' + ('noGSR_all' if modality == 'rest' else 'all') + '.csv')
    if bias_path.is_file():
        bias = u07_pd.read_csv(bias_path)
        labels = bias.connection.map({'shared_core':'Shared core','stroop_priority':'Stroop integration','gonogo_priority':'Go/No-Go integration','bridge':'Bridge'}).tolist() if modality == 'rest' else [compact_task_names[r.task] + ' | ' + compact_roi_names[r.roi_id] for r in bias.itertuples()]
        qcolumn = 'q_FDR_4' if modality == 'rest' else 'q_FDR_5_per_task'
        labels = [label + '\n' + f'(n={int(n)}, q={q:.3f})' for label, n, q in zip(labels, bias.N, bias[qcolumn])]
        (fig, ax) = plt.subplots(figsize=(3.15, 1.75) if modality == 'rest' else (4.1, 3.1))
        yy = u07_np.arange(len(bias))
        ax.errorbar(bias.beta, yy, xerr=[bias.beta - bias.CI_low, bias.CI_high - bias.beta], fmt='o', color='#246785', capsize=2.2, markersize=3, linewidth=.85)
        ax.axvline(0, color='.5', lw=0.8)
        ax.set(yticks=yy, yticklabels=labels, xlabel='Go/No-Go z: standardized beta and pointwise 95% CI')
        ax.invert_yaxis()
        ax.set_title('Starting-point bias z: resting connectivity' if modality == 'rest' else 'Starting-point bias z: task ROI associations')
        add_estimate_labels(ax, bias.beta, bias.CI_high, yy)
        export(fig, 'starting_point_bias_' + ('rest_connectivity' if modality == 'rest' else 'task_activation'))
    if modality == 'rest':
        a = u07_pd.read_csv(folder / 'tables' / (prefix + 'rest_shared_core_and_bridge_parameter_regression_noGSR_all.csv'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        b = u07_pd.read_csv(folder / 'tables' / (prefix + 'rest_priority_integration_parameter_regression_noGSR_all.csv'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        d = u07_pd.concat([a, b])
        mat = d.pivot(index='outcome', columns='parameter', values='beta').reindex(index=u07_hddm_FC, columns=u07_hddm_TERMS)
        q = d.pivot(index='outcome', columns='parameter', values='q_FDR_10').reindex(index=u07_hddm_FC, columns=u07_hddm_TERMS)
        (fig, ax) = plt.subplots(figsize=(3.6, 2.25))
        lim = max(0.2, abs(mat.to_numpy()).max())
        im = ax.imshow(mat, cmap='RdBu_r', norm=TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim), aspect='auto')
        ax.set_xticks(range(5), ['Shared v', 'Shared a', 'Shared t', 'Stroop\nresidual', 'Go/No-Go\nresidual'], rotation=0, ha='center')
        ax.set_yticks(range(4), ['Shared core', 'Stroop integration', 'Go/No-Go integration', 'Bridge'])
        for i in range(4):
            for j in range(5):
                ax.text(j, i, f'{mat.iloc[i, j]:.2f}' + ('*' if q.iloc[i, j] < 0.05 else ''), ha='center', va='center', fontsize=6.2)
        ax.set_title('')
        ax.set_xlabel('')
        fig.colorbar(im, ax=ax, label='Standardized beta', shrink=0.85, fraction=0.035, pad=0.035)
        export(fig, 'rest_connectivity_and_computational_components')
        d = u07_pd.read_csv(folder / 'tables' / (prefix + 'rest_priority_connectivity_direct_coefficient_difference_noGSR_all.csv'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        (fig, ax) = plt.subplots(figsize=(4.2, 1.8))
        yy = u07_np.arange(3)
        ax.errorbar(d.delta_beta, yy, xerr=[d.delta_beta - d.CI_low, d.CI_high - d.delta_beta], fmt='o', capsize=3, color='#246785')
        ax.axvline(0, color='.5', lw=0.8)
        ax.set_yticks(yy, ['Stroop: matched minus cross', 'Go/No-Go: matched minus cross', 'Sum of matched-minus-cross contrasts'])
        ax.invert_yaxis()
        ax.set_xlabel('Difference in standardized beta (95% paired bootstrap CI)')
        add_estimate_labels(ax, d.delta_beta, d.CI_high, yy)
        export(fig, 'rest_connectivity_relative_specificity_direct_test')
    else:
        d = u07_pd.read_csv(folder / 'tables' / (prefix + 'task_five_ROI_hierarchical_regression_coefficients_all_shared_general.csv'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        pick = d[d.roi_id.isin(u07_hddm_ROIS[:3]) & d.parameter.eq('shared_v') | d.roi_id.eq('stroop_lIPS') & d.task.eq('stroop') & d.parameter.eq('stroop_specific') | d.roi_id.eq('gonogo_rIFG_PMC') & d.task.eq('gonogo') & d.parameter.eq('gonogo_specific')].copy()
        labels = [f'{compact_task_names[r.task]} | {compact_roi_names[r.roi_id]} | {compact_term_names[r.parameter]}' for r in pick.itertuples()]
        (fig, ax) = plt.subplots(figsize=(4.25, 2.65))
        yy = u07_np.arange(len(pick))
        ax.errorbar(pick.beta, yy, xerr=[pick.beta - pick.CI_low, pick.CI_high - pick.beta], fmt='o', capsize=2.2, markersize=3, elinewidth=0.9, color='#246785')
        ax.axvline(0, color='.5', lw=0.8)
        ax.set_yticks(yy, labels, fontsize=7)
        ax.invert_yaxis()
        ax.set_xlabel('Standardized beta (95% CI); selected planned coefficients')
        add_estimate_labels(ax, pick.beta, pick.CI_high, yy, lower_bounds=pick.CI_low)
        export(fig, 'task_general_efficiency_and_priority_ROI_parameters')
        d = u07_pd.read_csv(folder / 'tables' / (prefix + 'task_five_ROI_specific_incremental_R2_all_shared_general.csv'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        (fig, ax) = plt.subplots(figsize=(3.9, 2.55))
        yy = u07_np.arange(len(d))
        ax.hlines(yy, d.CI_low, d.CI_high, color='#B56947', lw=0.9)
        ax.scatter(d.delta_R2, yy, color='#B56947', s=11)
        ax.set_yticks(yy, [compact_task_names[r.task] + ' | ' + compact_roi_names[r.roi_id] for r in d.itertuples()], fontsize=7)
        ax.invert_yaxis()
        ax.axvline(0, color='.5', lw=0.8)
        ax.set_xlabel('Added R² from both relative task components (95% bootstrap CI)')
        add_estimate_labels(ax, d.delta_R2, d.CI_high, yy)
        export(fig, 'task_specific_component_incremental_variance_explained')
        display_names = {'shared_SMA': 'SMA', 'shared_rAI_FO': 'Right AI/FO', 'shared_lAI': 'Left AI'}
        if (folder / 'tables' / (prefix + 'exploratory_nested_prediction_summary_all.csv')).is_file():
            d = u07_pd.read_csv(folder / 'tables' / (prefix + 'exploratory_nested_prediction_summary_all.csv'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            d = d[d.metric.eq('R2')].copy()
            labels = []
            for name in d.model:
                area = next((r for r in u07_hddm_ROIS[:3] if name.startswith(r + '_')))
                direction = name[len(area) + 1:].replace('_covariates_only', '').replace('_to_', ' → ')
                labels.append(display_names[area] + ' | ' + direction + (' | covariates' if name.endswith('_covariates_only') else ' | full'))
            (fig, ax) = plt.subplots(figsize=(4.0, max(2.2, 0.23 * len(d))))
            yy = u07_np.arange(len(d))
            ax.hlines(yy, d.split_p025, d.split_p975, color='#246785', lw=0.9)
            ax.scatter(d['mean'], yy, s=12, color='#246785')
            ax.axvline(0, color='.5', lw=0.8)
            ax.set_yticks(yy, labels, fontsize=8)
            ax.invert_yaxis()
            ax.set_xlabel('Out-of-fold R²: mean and 2.5–97.5% split range (not a sampling CI)')
            export(fig, 'shared_ROI_bidirectional_cross_task_generalization')
        increment_path = folder / 'tables' / (prefix + 'prediction_increment_covariate_control_all.csv')
        if increment_path.is_file():
            inc = u07_pd.read_csv(increment_path, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            inc = inc[inc.cross_task.eq(True)]
            (fig, ax) = plt.subplots(figsize=(4.0, 2.3))
            yy = u07_np.arange(len(inc))
            ax.hlines(yy, inc.Bonferroni_6_low, inc.Bonferroni_6_high, color='#B9C8CF', lw=2.1)
            ax.hlines(yy, inc.CI95_low, inc.CI95_high, color='#246785', lw=0.9)
            ax.scatter(inc.delta_R2, yy, color='#246785', s=12)
            ax.axvline(0, color='.5', lw=0.8)
            ax.set(yticks=yy, yticklabels=inc.model, xlabel='OOF R² increment over covariates; fixed-prediction bootstrap intervals')
            export(fig, 'shared_ROI_cross_task_prediction_increment')
        d = u07_pd.read_csv(folder / 'tables' / (prefix + 'cross_task_specific_component_predicts_other_task_activation_all.csv'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
        labels = [('Stroop' if r.predictor_task == 'stroop' else 'Go/No-Go') + ' residual → ' + ('Stroop' if r.target_task == 'stroop' else 'Go/No-Go') + ' | ' + display_names[r.roi_id] for r in d.itertuples()]
        (fig, ax) = plt.subplots(figsize=(4.1, 3.0))
        yy = u07_np.arange(len(d))
        ax.axvspan(-0.3, 0.3, color='#246785', alpha=0.07)
        ax.axvline(-0.3, color='#777777', lw=0.8, ls='--')
        ax.axvline(0.3, color='#777777', lw=0.8, ls='--')
        ax.axvline(0, color='#999999', lw=0.7)
        ax.hlines(yy, d.r90_low, d.r90_high, color='#246785', lw=0.9)
        ax.scatter(d.partial_r, yy, s=11, color='#246785')
        ax.set_yticks(yy, labels, fontsize=7)
        ax.invert_yaxis()
        ax.set_xlim(min(-0.4, d.r90_low.min() - 0.05), max(0.4, d.r90_high.max() + 0.05))
        ax.set_xlabel('Partial r and 90% CI; equivalence bounds ±.30')
        add_estimate_labels(ax, d.partial_r, u07_np.maximum(d.r90_high, 0.33), yy)
        export(fig, 'shared_ROI_null_effect_estimates_and_equivalence_intervals')


In [ ]:
def u07_hddm_starting_bias_sensitivity(bundle, modality, cfg=u07_hddm_CFG):
    allrows = []
    for (variant, d) in u07_hddm_variants(bundle, modality):
        rows = []
        if modality == 'task':
            groups = [(dict(task=task, roi_id=roi), g, 'mean_voxel_t') for ((task, roi), g) in d.groupby(['task', 'roi_id'])]
        else:
            groups = [(dict(connection=outcome), d, outcome) for outcome in u07_hddm_FC]
        for (keys, g, outcome) in groups:
            fit = u07_hddm_model(g, outcome, u07_hddm_COV + u07_hddm_TERMS + ['gonogo_z'])
            rows.append(dict(**keys, **u07_hddm_coefficients(fit, ['gonogo_z'], cfg)[0]))
        tab = u07_pd.DataFrame(rows)
        if modality == 'task':
            tab['q_FDR_5_per_task'] = tab.groupby('task').p.transform(lambda p: u07_hddm_bh(p))
        else:
            tab['q_FDR_4'] = u07_hddm_bh(tab.p)
        tab['variant'] = variant
        u07_hddm_save(tab, u07_hddm_TASK if modality == 'task' else u07_hddm_REST, 'starting_point_bias_sensitivity_' + variant, bundle['status'])
        allrows.append(tab)
    return u07_pd.concat(allrows, ignore_index=True)


In [ ]:
def u10_brain_behavior_behavior():
    subject_scores = u00_behavior_subject_scores
    participants = u15_participants
    return participants()[['subject_id', 'age', 'sex', 'education']].merge(subject_scores(), on='subject_id', validate='one_to_one')


In [ ]:
import matplotlib.pyplot as u10_plt
import matplotlib as u10_matplotlib
u10_matplotlib.use('Agg')
import matplotlib.pyplot as u10_plt
import matplotlib.pyplot as u10_plt
from pathlib import Path as u15_Path
import os as u15_os, hashlib as u15_hashlib, json as u15_json
from importlib import metadata as u15_metadata
import platform as u15_platform, sys as u15_sys
import numpy as u15_np
import pandas as u15_pd
u15_ROOT = ANALYSIS_ROOT
u15_DATA = u15_Path(u15_os.environ.get('A8_DATA_DIR', NOTEBOOK_PATH.parent)).expanduser().resolve()
u15_RESULTS = u15_Path(u15_os.environ.get('A8_OUTPUT_DIR', u15_ROOT / 'results')).expanduser().resolve()
u15_EXPECTED_N = int(u15_os.environ.get('A8_EXPECTED_N', '106'))
u15_PARAMS = ['stroop_a', 'stroop_v_congruent', 'stroop_v_neutral', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u15_INPUT_FOLDERS = {'MRI_manifest.csv': '7.task_fmri', 'Participants.csv': '1.behavior', 'ROI_masks.csv': '7.task_fmri', 'HDDM_posterior_draws.csv': '2.hddm_fitting', 'HDDM_parameters.csv': '2.hddm_fitting', 'Behavior_data.csv': '1.behavior', 'HDDM_data.csv': '2.hddm_fitting'}
u15_INPUT_FOLDERS.update({'MRI_preprocessing.csv': '7.task_fmri', 'MultiEcho_runs.csv': '7.task_fmri'})
u15_INPUT_FILES = {name: name for name in u15_INPUT_FOLDERS}
u15_INPUT_COLUMNS = {'Participants.csv': ('subject_id', 'age', 'sex', 'education'), 'Behavior_data.csv': ('subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'), 'HDDM_data.csv': ('subj_idx', 'task', 'run', 'trial_id', 'condition', 'stimulus', 'response', 'rt'), 'MRI_manifest.csv': ('subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'), 'ROI_masks.csv': ('roi_id', 'path', 'source_doi', 'independent_of_sample'), 'HDDM_parameters.csv': ('subject_id', *u15_PARAMS), 'HDDM_posterior_draws.csv': ('task', 'chain', 'draw', 'subject_id', 'parameter', 'value')}


In [ ]:
def u15_aggregate_task_motion(motion):
    required = ['subject_id', 'task', 'mean_fd_mm', 'fd_gt_05_fraction', 'all_spikes_fraction', 'n_volumes']
    if set(required) - set(motion):
        raise ValueError('Task motion table is missing columns')
    rows = []
    for ((sid, task), g) in motion.groupby(['subject_id', 'task']):
        values = g[required[2:]].to_numpy(float)
        if not u15_np.isfinite(values).all() or g.n_volumes.le(0).any():
            raise ValueError('Invalid observed run motion statistics')
        strict = ((g.mean_fd_mm < 0.2) & (g.fd_gt_05_fraction < 0.1) & (g.all_spikes_fraction < 0.1)).all()
        rows.append(dict(subject_id=sid, task=task, mean_fd_mm=u15_np.average(g.mean_fd_mm, weights=g.n_volumes), strict_motion=bool(strict)))
    return u15_pd.DataFrame(rows)


In [ ]:
def u15_input_path(name):
    filename = u15_INPUT_FILES.get(name, name)
    p = u15_DATA / filename
    if not p.is_file() and 'A8_DATA_DIR' not in u15_os.environ:
        p = u15_ROOT / u15_INPUT_FOLDERS.get(name, '7.task_fmri') / filename
    if name == 'MRI_manifest.csv' and (u15_os.environ.get('A8_MRI_INPUT_MODE') == 'generated' or not p.is_file()):
        p = u15_RESULTS / '7.task_fmri/tables/MRI_manifest.csv'
    fitted = (u15_RESULTS / '2.hddm_fitting/fit_state.json').is_file() and u15_os.environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external'
    if name in {'HDDM_parameters.csv', 'HDDM_posterior_draws.csv'} and (fitted or not p.is_file()):
        p = u15_RESULTS / '2.hddm_fitting/tables' / name
    if not p.is_file():
        raise FileNotFoundError(f'Required observed input missing: {p}. Supply the required real input file; no example values are substituted.')
    return p


In [ ]:
def u15_table(name, columns=(), unique=None):
    p = u15_input_path(name)
    d = u15_pd.read_csv(p, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    if d.empty:
        raise ValueError(f'{p.name}: no observed records; a header-only template is not data')
    missing = (set(columns) | set(u15_INPUT_COLUMNS.get(name, ()))) - set(d)
    if missing:
        raise ValueError(f'{p.name}: missing columns {sorted(missing)}')
    if unique and (d[list(unique)].isna().any().any() or d.duplicated(list(unique)).any()):
        raise ValueError(f'{p.name}: missing/duplicate key {unique}')
    for c in ('subject_id', 'subj_idx', 'trial_id'):
        if c in d and (d[c].isna().any() or d[c].str.strip().eq('').any() or d[c].ne(d[c].str.strip()).any()):
            raise ValueError(f'{p.name}: blank or whitespace-padded identifier in {c}')
    for c in ('data_status', 'source_status'):
        if c in d and d[c].astype(str).str.contains('synthetic|simulation|not empirical', case=False, regex=True).any():
            raise ValueError(f'{p.name}: simulated rows cannot enter this workflow')
    return d


In [ ]:
def u15_participants():
    d = u15_table('Participants.csv', ['subject_id', 'age', 'sex', 'education'], ['subject_id'])
    if len(d) != u15_EXPECTED_N:
        raise ValueError(f'Expected {u15_EXPECTED_N} participants, found {len(d)}; configure A8_EXPECTED_N deliberately.')
    for c in ['age', 'sex', 'education']:
        d[c] = u15_pd.to_numeric(d[c], errors='raise')
        if not u15_np.isfinite(d[c]).all():
            raise ValueError(f'Nonfinite {c}')
    if not d.sex.isin([0, 1]).all():
        raise ValueError('sex must use the documented 0/1 coding')
    return d.sort_values('subject_id').reset_index(drop=True)


In [ ]:
def u15_match_cohort(d, name, keys=('subject_id',), complete=True):
    ids = set(u15_participants().subject_id)
    if d.subject_id.isna().any() or not set(d.subject_id) <= ids:
        raise ValueError(f'{name}: unexpected or missing subject_id')
    if complete and set(d.subject_id) != ids:
        raise ValueError(f'{name}: cohort mismatch; do not fill missing participants')
    if d.duplicated(list(keys)).any():
        raise ValueError(f'{name}: duplicate {keys}')
    return d


In [ ]:
def u15_behavior():
    d = u15_table('Behavior_data.csv', ['subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'], ['subject_id', 'task', 'run', 'trial_id'])
    u15_match_cohort(d, 'Behavior_data', ('subject_id', 'task', 'run', 'trial_id'))
    if not d.task.isin(['stroop', 'gonogo']).all():
        raise ValueError('task must be stroop/gonogo')
    d['rt_s'] = u15_pd.to_numeric(d.rt_s, errors='raise')
    d['response'] = u15_pd.to_numeric(d.response, errors='raise')
    d['correct'] = u15_pd.to_numeric(d.correct, errors='raise')
    if not d.correct.isin([0, 1]).all() or not d.response.dropna().isin([0, 1]).all():
        raise ValueError('correct and observed response must be binary')
    if not u15_np.isfinite(d.rt_s.dropna()).all() or (d.rt_s.dropna() <= 0).any():
        raise ValueError('RTs must be in seconds; nonresponses have NaN RT')
    for (task, conds) in [('stroop', {'congruent', 'neutral', 'incongruent'}), ('gonogo', {'go', 'nogo'})]:
        g = d[d.task.eq(task)]
        if set(g.subject_id) != set(u15_participants().subject_id) or set(g.condition) != conds:
            raise ValueError(f'{task}: incomplete subjects/conditions')
        if not g.groupby('subject_id').condition.apply(lambda x: set(x) == conds).all():
            raise ValueError(f'{task}: a participant is missing a condition')
    return d


In [ ]:
def u15_posterior():
    d = u15_table('HDDM_parameters.csv', ['subject_id'] + u15_PARAMS, ['subject_id'])
    u15_match_cohort(d, 'HDDM_parameters')
    if not u15_np.isfinite(d[u15_PARAMS].to_numpy(float)).all():
        raise ValueError('Nonfinite fitted parameter')
    for c in ['stroop_a', 'gonogo_a', 'stroop_t', 'gonogo_t']:
        if (d[c] <= 0).any():
            raise ValueError(f'Parameter outside domain: {c}')
    if not d.gonogo_z.between(0, 1, inclusive='neither').all():
        raise ValueError('gonogo_z outside (0,1)')
    u15_verify_fitted_artifact(u15_input_path('HDDM_parameters.csv'))
    u15_provenance('posterior_input', ['HDDM_parameters.csv', 'Participants.csv'])
    return d


In [ ]:
def u15_source_signature():
    document = u15_json.loads(NOTEBOOK_PATH.read_text())
    source = '\n'.join((''.join(cell['source']) for cell in document['cells'] if cell['cell_type'] == 'code'))
    return u15_hashlib.sha256(source.encode()).hexdigest()


In [ ]:
def u15_runtime_versions():
    versions = {'python': u15_sys.version, 'platform': u15_platform.platform()}
    for name in ['numpy', 'scipy', 'pandas', 'statsmodels', 'scikit-learn', 'nilearn', 'nibabel', 'arviz', 'hddm', 'kabuki', 'pymc', 'matplotlib', 'openpyxl', 'antspyx', 'templateflow']:
        try:
            versions[name] = u15_metadata.version(name)
        except u15_metadata.PackageNotFoundError:
            versions[name] = None
    return versions


In [ ]:
def u15_atomic_record(path, record):
    path = u15_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.pending')
    temp.write_text(u15_json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


In [ ]:
def u15_provenance(stage, names, settings=None):
    records = []
    for name in names:
        path = u15_input_path(name)
        records.append({'file': str(path), 'sha256': u15_hashlib.sha256(path.read_bytes()).hexdigest()})
    record = dict(stage=stage, input_files=records, code_sha256=u15_source_signature(), runtime=u15_runtime_versions(), settings=settings or {})
    u15_atomic_record(u15_RESULTS / (stage + '_inputs.json'), record)
    return record


In [ ]:
def u15_verify_fitted_artifact(path):
    path = u15_Path(path).resolve()
    generated = (u15_RESULTS / '2.hddm_fitting/tables').resolve()
    if path.parent != generated:
        return
    state_path = generated.parent / 'fit_state.json'
    if not state_path.is_file():
        raise RuntimeError('Generated HDDM tables lack a completed fit record; rerun the fit or use explicitly supplied external estimates')
    state = u15_json.loads(state_path.read_text())
    if state.get('model_code_sha256') != u15_hddm_source_signature():
        raise RuntimeError('HDDM model implementation changed since fitting; refit before using generated estimates')
    if state.get('status') != 'complete':
        raise RuntimeError('HDDM fit is incomplete; downstream analyses cannot read its tables')
    for (name, expected) in state['input_sha256'].items():
        if u15_hashlib.sha256(u15_input_path(name).read_bytes()).hexdigest() != expected:
            raise RuntimeError('Observed inputs changed since HDDM fitting: ' + name)
    diagnostics = generated / 'HDDM_sampling_diagnostics.csv'
    if path.name == 'HDDM_parameters.csv':
        if not diagnostics.is_file() or diagnostics.stat().st_mtime_ns < state_path.stat().st_mtime_ns:
            raise RuntimeError('Run chain diagnostics for the completed fit before associations')
        check = u15_pd.read_csv(diagnostics, dtype={'subject_id': str})
        if check.empty or not check.diagnostics_pass.astype(str).str.lower().eq('true').all():
            raise RuntimeError('HDDM sampling diagnostics did not all pass')


In [ ]:
def u15_observed_hddm_data():
    d = u15_table('HDDM_data.csv', unique=['subj_idx', 'task', 'run', 'trial_id']).rename(columns={'subj_idx': 'subject_id'})
    raw = u15_behavior()
    keep = raw.task.eq('gonogo') | raw.task.eq('stroop') & raw.rt_s.notna() & raw.response.notna()
    raw = raw.loc[keep]
    keys = ['subject_id', 'task', 'run', 'trial_id']
    if set(map(tuple, d[keys].to_numpy())) != set(map(tuple, raw[keys].to_numpy())):
        raise ValueError('HDDM and behavioral trial sets differ; retain observed Stroop responses and every Go/No-Go trial')
    joined = d.merge(raw[keys + ['condition', 'rt_s', 'response', 'correct']], on=keys, suffixes=('', '_behavior'), validate='one_to_one')
    if not joined.condition.eq(joined.condition_behavior).all():
        raise ValueError('HDDM and behavioral condition coding differs')
    if not joined.response.isin([0, 1]).all() or not u15_np.isfinite(joined.rt).all():
        raise ValueError('Invalid HDDM response or RT')
    st = joined.task.eq('stroop')
    if not joined.loc[st, 'stimulus'].isin([0, 1]).all() or not joined.loc[st, 'response'].eq(joined.loc[st, 'response_behavior']).all():
        raise ValueError('Stroop stimulus/response coding mismatch')
    if not joined.loc[st, 'correct'].eq(joined.loc[st, 'response'].eq(joined.loc[st, 'stimulus']).astype(int)).all():
        raise ValueError('Stroop correctness differs from the color-boundary coding')
    ng = ~st
    observed = joined.response.eq(1) | st
    if not u15_np.allclose(joined.loc[observed, 'rt'], joined.loc[observed, 'rt_s'], rtol=1e-08, atol=1e-09):
        raise ValueError('HDDM RTs differ from observed behavioral RTs')
    if not joined.loc[ng, 'response'].eq(joined.loc[ng, 'response_behavior']).all():
        raise ValueError('Go/No-Go response coding mismatch')
    if not joined.loc[ng & ~observed, 'rt'].eq(999).all() or joined.loc[ng & ~observed, 'rt_s'].notna().any():
        raise ValueError('Go/No-Go nonresponses must have blank behavioral RT and HDDM RT 999')
    return d.rename(columns={'subject_id': 'subj_idx'})


In [ ]:
def u15_hddm_source_signature():
    import types as types
    fit = u01_ddm_fit_hddm
    backend = u01_ddm_validate_hddm_backend
    sample = u01_ddm_recovery_sample
    predictive = u01_ddm_posterior_predictive_samples
    import re as re

    def normalized(code):
        return [code.co_code.hex(), [normalized(x) if isinstance(x, types.CodeType) else repr(x) for x in code.co_consts], [re.sub('^u[0-9]+_', '', x) for x in code.co_names], list(code.co_varnames), code.co_argcount, code.co_kwonlyargcount]
    payload = [normalized(f.__code__) for f in [fit, backend, sample, predictive, u15_observed_hddm_data]]
    return u15_hashlib.sha256(u15_json.dumps(payload, sort_keys=True).encode()).hexdigest()


In [ ]:
import numpy as u16_np
from scipy import stats as u16_stats, integrate as u16_integrate


In [ ]:
def u16_equivalence(r, n, k, bound=0.3):
    if not 0 < bound < 1 or n <= k + 3:
        raise ValueError('Invalid equivalence specification')
    zz = u16_np.arctanh(u16_np.clip(r, -1 + 1e-12, 1 - 1e-12))
    se = 1 / u16_np.sqrt(n - k - 3)
    limit = u16_np.arctanh(bound)
    ci = u16_np.tanh(zz + u16_np.array([-1, 1]) * u16_stats.norm.ppf(0.95) * se)
    return dict(bound_r=bound, r90_low=ci[0], r90_high=ci[1], p_TOST=max(u16_stats.norm.sf((zz + limit) / se), u16_stats.norm.cdf((zz - limit) / se)))


In [ ]:
def u16_jzs_nested(y, X_full, X_null):
    y = u16_np.asarray(y, float)
    n = len(y)
    sst = u16_np.sum((y - y.mean()) ** 2)
    if sst <= 0:
        raise ValueError('Constant outcome')
    designs = [u16_np.c_[u16_np.ones(n), u16_np.asarray(v, float)] for v in [X_full, X_null]]
    if any((u16_np.linalg.matrix_rank(X) != X.shape[1] for X in designs)):
        raise ValueError('JZS regression requires full-rank designs')
    if not u16_np.allclose(designs[1], designs[0] @ u16_np.linalg.lstsq(designs[0], designs[1], rcond=None)[0]):
        raise ValueError('Null model is not nested in alternative')
    values = []
    for X in designs:
        p = X.shape[1] - 1
        if n <= p + 2:
            raise ValueError('Insufficient observations for JZS regression')
        err = y - X @ u16_np.linalg.lstsq(X, y, rcond=None)[0]
        r2 = float(u16_np.clip(1 - u16_np.sum(err ** 2) / sst, 0, 1 - 1e-14))

        def logfun(u):
            return (n - 1 - p) / 2 * u16_np.logaddexp(0, u) - (n - 1) / 2 * u16_np.logaddexp(0, u16_np.log1p(-r2) + u) - 0.5 * u - n / 2 * u16_np.exp(-u)
        grid = u16_np.linspace(-25, 60, 400)
        shift = max((logfun(u) for u in grid))
        integral = u16_integrate.quad(lambda u: u16_np.exp(logfun(u) - shift), -30, 80, epsabs=1e-10, epsrel=1e-09, limit=300)[0]
        if integral <= 0 or not u16_np.isfinite(integral):
            raise ArithmeticError('JZS integration failed')
        values.append(u16_np.log(integral) + shift)
    bf = float(u16_np.exp(u16_np.clip(values[0] - values[1], -700, 700)))
    return dict(BF10=bf, BF01=1 / bf, JZS_prior='g ~ inverse-gamma(1/2, n/2); Wetzels & Wagenmakers (2012), Eq. 15')


In [ ]:
from pathlib import Path as u17_Path
import json as u17_json
import os as u17_os
for u17__key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u17_os.environ.setdefault(u17__key, '1')
import numpy as u17_np
import pandas as u17_pd
import nilearn as u17_nilearn
if u17_nilearn.__version__ != '0.13.1':
    raise RuntimeError('Use Nilearn 0.13.1, as specified in the manuscript')
from pathlib import Path as u17_Path
u17_RESULTS = u15_RESULTS
u17_participants = u15_participants
u17_MRI = u17_RESULTS / '3.mri_analysis'
u17_TASK = u17_MRI / '2.task_fmri'
u17_REST = u17_MRI / '1.resting_state'
u17_NEW = u17_TASK / 'other_outputs'
u17_REST_NEW = u17_REST / 'other_outputs'
u17_ROIS = ['shared_SMA', 'shared_rAI_FO', 'shared_lAI', 'stroop_lIPS', 'gonogo_rIFG_PMC']
u17_CONTRAST = {'stroop': 'incongruent_minus_congruent', 'gonogo': 'nogo_correct_minus_go_correct'}


In [ ]:
def u17_subjects():
    return u17_participants().subject_id.tolist()


In [ ]:
def u17_atomic_json(path, data):
    path = u17_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')

    def native(value):
        if isinstance(value, u17_np.generic):
            return value.item()
        if isinstance(value, u17_np.ndarray):
            return value.tolist()
        raise TypeError(f'Unsupported metadata type: {type(value).__name__}')
    tmp.write_text(u17_json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False, default=native))
    tmp.replace(path)


In [ ]:
u19_np = u17_np
u19_pd = u17_pd
u19_NEW = u17_NEW
u19_REST_NEW = u17_REST_NEW
u19_ROIS = u17_ROIS
u19_CONTRAST = u17_CONTRAST
u19_subjects = u17_subjects
u19_atomic_json = u17_atomic_json


In [ ]:
def u19_main():
    ids = set(u19_subjects())
    a = u19_pd.read_csv(u19_NEW / 'independent_ALE_ROI_ALE_3D_mean_t.csv', dtype={'subject_id': str})
    expected = {(sid, task, roi) for sid in ids for task in u19_CONTRAST for roi in u19_ROIS}
    if a.duplicated(['subject_id', 'task', 'roi_id']).any() or set(map(tuple, a[['subject_id', 'task', 'roi_id']].to_numpy())) != expected:
        raise ValueError('Task ROI table is missing or duplicating participant/task/ROI keys')
    if not a.coverage.between(0, 1).all() or not u19_np.array_equal(a.mean_voxel_t.notna(), a.coverage.ge(0.8)):
        raise ValueError('Task ROI coverage and missing-value flags disagree')
    if not u19_np.isfinite(a.mean_voxel_t.dropna()).all():
        raise ValueError('Nonfinite task activation')
    p = u19_pd.read_csv(u19_NEW / 'independent_ALE_ROI_unsmoothed_patterns.csv', dtype={'subject_id': str})
    expected = {(sid, roi) for sid in ids for roi in u19_ROIS[:3]}
    if p.duplicated(['subject_id', 'roi_id']).any() or set(map(tuple, p[['subject_id', 'roi_id']].to_numpy())) != expected:
        raise ValueError('Pattern table participant/ROI keys are incomplete')
    if not p.coverage.between(0, 1).all() or not u19_np.array_equal(p.fisher_z.notna(), p.coverage.ge(0.8)) or (not u19_np.isfinite(p.fisher_z.dropna()).all()):
        raise ValueError('Pattern coverage or finite-value checks failed')
    fc = u19_pd.read_csv(u19_REST_NEW / 'independent_ALE_ROI_connectivity_subject_level.csv', dtype={'subject_id': str})
    flags = fc.GSR.astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False})
    if flags.isna().any():
        raise ValueError('Invalid GSR flag')
    fc['GSR'] = flags
    expected = {(sid, gsr) for sid in ids for gsr in [False, True]}
    if fc.duplicated(['subject_id', 'GSR']).any() or set(map(tuple, fc[['subject_id', 'GSR']].to_numpy())) != expected:
        raise ValueError('Resting connectivity participant/GSR keys are incomplete')
    cols = ['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge']
    if u19_np.isinf(fc[cols].to_numpy(float)).any():
        raise ValueError('Infinite resting connectivity')
    result = dict(status='computed_artifact_checks_passed', subjects=len(ids), pattern_N=p.groupby('roi_id').fisher_z.count().to_dict(), strict_rest_N=int(fc.loc[fc.strict_motion.astype(str).str.lower().isin(['true', '1']), 'subject_id'].nunique()))
    u19_atomic_json(u19_NEW / 'final_validation.json', result)
    return result


In [ ]:
u00_behavior = u15_behavior
u01_RESULTS = u15_RESULTS
u01_hddm_source_signature = u15_hddm_source_signature
u01_RESULTS = u15_RESULTS
u01_participants = u15_participants
u01_input_path = u15_input_path
u01_provenance = u15_provenance
u01_PARAMS = u15_PARAMS
u01_observed_hddm_data = u15_observed_hddm_data
u01_atomic_record = u15_atomic_record
u01_ddm_OUT = u01_RESULTS / '2.hddm_fitting'
u07_ROOT = u15_ROOT
u07_DATA = u15_DATA
u07_RESULTS = u15_RESULTS
u07_sys.dont_write_bytecode = True
u07_jzs_nested = u16_jzs_nested
u07_equivalence = u16_equivalence
u07_PROJECT = u15_ROOT
u07_RESULTS = u15_RESULTS
u07_DATA = u15_DATA
u07_posterior = u15_posterior
u07_EXPECTED_N = u15_EXPECTED_N
u07_hddm_DDM = u07_RESULTS
u07_hddm_TASK = u07_hddm_DDM / '5.task_hddm'
u07_hddm_REST = u07_hddm_DDM / '4.resting_hddm'
u07_hddm_MRI = u07_RESULTS / '3.mri_analysis'
u10_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u10_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})


In [ ]:
def a8_display_three_line(value, *args, **kwargs):
    import pandas as pd
    from IPython.display import display, HTML
    if isinstance(value, pd.DataFrame):
        html = value.to_html(index=False, border=0, escape=True)
        html = html.replace('<table ', '<table style="border-collapse:collapse;border-top:2px solid black;border-bottom:2px solid black;" ', 1)
        html = html.replace('<th>', '<th style="border:0;border-bottom:1px solid black;padding:7px 16px;text-align:center;">')
        html = html.replace('<td>', '<td style="border:0;padding:7px 16px;text-align:center;">')
        return display(HTML(html), *args, **kwargs)
    return display(value, *args, **kwargs)
u07_display = a8_display_three_line


In [ ]:
u07_CFG = u07_hddm_CFG
u07_bundle = u07_hddm_load_real_bundle()


In [ ]:
for u07_modality_name in ['task', 'rest', 'pattern']:
    print(u07_modality_name, u07_bundle[u07_modality_name].subject_id.nunique(), len(u07_bundle[u07_modality_name]))


In [ ]:
u07_display(u07_hddm_task_primary(u07_bundle, u07_CFG))
u07_display(u07_hddm_task_secondary(u07_bundle, u07_CFG))


In [ ]:
u07_display(u07_hddm_starting_bias_sensitivity(u07_bundle, 'task', u07_CFG))


In [ ]:
u07_hddm_figures(u07_bundle, 'task', u07_CFG)
